# OMR Pipeline — Stage-by-Stage Output Collector

This notebook uses the existing `omr_bubble_detector.py` implementation to generate presentation-ready intermediate images from one phone photograph.

It follows the complete methodology pipeline:

1. Capture and image decoding
2. Frame detection and perspective correction
3. Printed-region detection
4. Bubble candidate detection
5. Grid inference and bubble assignment
6. Fill-score measurement and confidence
7. Student/Test ID decoding and answer grading
8. Final overlays, metadata, tables, JSON, and exported images

## How to use

1. Put this notebook in the same folder as `omr_bubble_detector.py` and `streamlit_omr_detector.py`.
2. Add a phone photograph of the answer sheet to that folder.
3. Change `IMAGE_PATH` in the configuration cell.
4. Run the notebook from top to bottom.
5. Collect the generated assets from the `omr_stage_outputs` folder.

The final detector results come from the real public functions. Small debug helpers reproduce the same parameters used by the implementation so normally hidden outputs—Canny edges, morphology masks, raw candidates, and grid support—can be exported for the presentation.

## 0. Environment and imports

The notebook automatically searches for the detector module in the current directory and `/mnt/data`.

In [ ]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_CANDIDATES = [Path.cwd(), Path('/mnt/data'), Path.cwd().parent]
PROJECT_DIR = next(
    (p for p in PROJECT_CANDIDATES if (p / 'omr_bubble_detector.py').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find omr_bubble_detector.py. Place this notebook beside the Python files.'
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import omr_bubble_detector as omr

print(f'Project directory: {PROJECT_DIR.resolve()}')
print(f'Detector module:    {Path(omr.__file__).resolve()}')
print(f'OpenCV version:     {cv2.__version__}')

## 1. Configuration

Change `IMAGE_PATH` before running the remaining cells.

- `EXPECTED_QUESTIONS = None` enables automatic row detection.
- Otherwise use `10`, `20`, `30`, `50`, or `100`.
- The answer key may be added as a dictionary, list, numbered text, JSON, or a plain A–E sequence.

In [ ]:
# REQUIRED: change this filename to your phone photograph.
IMAGE_PATH = PROJECT_DIR / 'your_phone_photo.jpg'

# None = automatic detection. Otherwise use 10, 20, 30, 50, or 100.
EXPECTED_QUESTIONS: Optional[int] = None

# None automatically selects a useful question for the fill-score example.
EXAMPLE_QUESTION: Optional[int] = None

# Answer-key options:
# ANSWER_KEY = {1: 'A', 2: 'C'}
# ANSWER_KEY = ['A', 'C', 'B']
# ANSWER_KEY_TEXT = '1:A\n2:C' or 'ABCDE...'
ANSWER_KEY: Any = {}
ANSWER_KEY_TEXT = ''
USE_DEMO_KEY = False

OUTPUT_DIR = PROJECT_DIR / 'omr_stage_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IMAGE_PATH.exists():
    available = sorted(
        p.name
        for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
        for p in PROJECT_DIR.glob(pattern)
    )
    raise FileNotFoundError(
        f'Image not found: {IMAGE_PATH}\n'
        f'Available images: {available or "none"}\n'
        'Update IMAGE_PATH in this cell.'
    )

print(f'Input image:   {IMAGE_PATH.resolve()}')
print(f'Output folder: {OUTPUT_DIR.resolve()}')

## Shared visualization and export helpers

In [ ]:
def bgr_to_rgb(image_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def save_image(filename: str, image: np.ndarray) -> Path:
    path = OUTPUT_DIR / filename
    ok = cv2.imwrite(str(path), image)
    if not ok:
        raise IOError(f'Could not save image: {path}')
    print(f'Saved: {path.name}')
    return path


def show_image(
    image: np.ndarray,
    title: str = '',
    figsize: Tuple[float, float] = (10, 8),
    cmap: Optional[str] = None,
) -> None:
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap=cmap or 'gray')
    else:
        plt.imshow(bgr_to_rgb(image))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_gallery(
    items: Sequence[Tuple[str, np.ndarray]],
    columns: int = 2,
    width: float = 7,
    height: float = 5,
) -> None:
    if not items:
        return
    rows = math.ceil(len(items) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(columns * width, rows * height))
    axes = np.array(axes, dtype=object).reshape(-1)
    for ax, (title, image) in zip(axes, items):
        if image.ndim == 2:
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(bgr_to_rgb(image))
        ax.set_title(title)
        ax.axis('off')
    for ax in axes[len(items):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def crop_roi(image: np.ndarray, roi: Sequence[int]) -> np.ndarray:
    x1, y1, x2, y2 = map(int, roi)
    return image[y1:y2, x1:x2].copy()


def draw_quad(
    image_bgr: np.ndarray,
    corners: Optional[np.ndarray],
    label: str,
    color: Tuple[int, int, int] = (0, 255, 255),
) -> np.ndarray:
    out = image_bgr.copy()
    if corners is None:
        cv2.putText(out, f'{label}: not detected', (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)
        return out
    pts = np.asarray(corners, dtype=np.float32).reshape(4, 2)
    pts_i = np.round(pts).astype(int)
    cv2.polylines(out, [pts_i.reshape(-1, 1, 2)], True, color, 4, cv2.LINE_AA)
    for name, (x, y) in zip(['TL', 'TR', 'BR', 'BL'], pts_i):
        cv2.circle(out, (int(x), int(y)), 10, color, -1)
        cv2.putText(out, name, (int(x) + 12, int(y) - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
    cv2.putText(out, label, (20, 35), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, color, 2, cv2.LINE_AA)
    return out


def draw_roi_overlay(warped_bgr: np.ndarray, layout: Dict[str, Any]) -> np.ndarray:
    out = warped_bgr.copy()
    styles = {
        'header': ((180, 0, 180), 'Header'),
        'student_id': ((255, 0, 0), 'Student ID'),
        'test_id': ((255, 170, 0), 'Test ID'),
        'answers': ((0, 170, 255), 'Answers'),
    }
    sources = layout.get('source', {})
    for key, (color, title) in styles.items():
        roi = layout.get(key)
        if not roi:
            continue
        x1, y1, x2, y2 = map(int, roi)
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 3)
        cv2.putText(out, f'{title}: {sources.get(key, "unknown")}',
                    (x1 + 4, max(24, y1 - 7)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, color, 2, cv2.LINE_AA)
    return out


def save_dataframe(df: pd.DataFrame, filename: str) -> Path:
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved: {path.name}')
    return path

# Stage 1 — Capture and Image Decoding

The phone photograph is read as an OpenCV BGR image. This exported image is the input visualization for the presentation.

In [ ]:
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise ValueError(f'OpenCV could not decode: {IMAGE_PATH}')

print(f'Image shape: {image_bgr.shape}')
print(f'Data type:   {image_bgr.dtype}')
save_image('01_original_phone_photo.png', image_bgr)
show_image(image_bgr, 'Stage 1 — Original Phone Photograph', figsize=(10, 8))

# Stage 2 — Frame Detection and Perspective Correction

Primary path:

- Grayscale conversion and Gaussian blur `(3 × 3)`
- Canny edge detection `(50, 150)`
- Probabilistic Hough line detection
- Horizontal/vertical orientation filtering
- Four fitted frame lines and their intersections
- Four-point perspective transform to `1000 × 1414`

The notebook also exports the Otsu and morphology images used by the physical-paper fallback.

In [ ]:
def frame_detection_debug(image_bgr: np.ndarray) -> Dict[str, Any]:
    h0, w0 = image_bgr.shape[:2]
    scale = 1000.0 / max(h0, w0)
    small = cv2.resize(image_bgr, (int(w0 * scale), int(h0 * scale)))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 50, 150)

    h, w = small.shape[:2]
    min_line = int(0.25 * min(h, w))
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, 70,
                            minLineLength=min_line, maxLineGap=30)
    all_lines = small.copy()
    oriented = small.copy()
    h_count = v_count = 0
    if lines is not None:
        for x1, y1, x2, y2 in lines[:, 0, :]:
            cv2.line(all_lines, (x1, y1), (x2, y2), (0, 255, 255), 2, cv2.LINE_AA)
            dx, dy = float(x2 - x1), float(y2 - y1)
            length = math.hypot(dx, dy)
            if length < min_line:
                continue
            angle = math.degrees(math.atan2(dy, dx))
            if abs(angle) < 12 or abs(abs(angle) - 180) < 12:
                h_count += 1
                cv2.line(oriented, (x1, y1), (x2, y2), (255, 0, 0), 3, cv2.LINE_AA)
            elif abs(abs(angle) - 90) < 12:
                v_count += 1
                cv2.line(oriented, (x1, y1), (x2, y2), (0, 140, 255), 3, cv2.LINE_AA)
    return {
        'small': small, 'gray': gray, 'blur': blur, 'edges': edges,
        'all_lines': all_lines, 'oriented': oriented,
        'frame_corners': omr._find_printed_frame_corners(image_bgr),
        'horizontal_count': h_count, 'vertical_count': v_count,
    }


def paper_contour_debug(image_bgr: np.ndarray) -> Dict[str, Any]:
    h0, w0 = image_bgr.shape[:2]
    scale = 900.0 / max(h0, w0)
    small = cv2.resize(image_bgr, (int(w0 * scale), int(h0 * scale)))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (11, 11))
    closed = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel, iterations=2)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel, iterations=1)
    return {
        'small': small, 'gray': gray, 'blur': blur, 'otsu': otsu,
        'closed': closed, 'opened': opened,
        'paper_corners': omr._find_page_by_contour(image_bgr),
    }


frame_debug = frame_detection_debug(image_bgr)
paper_debug = paper_contour_debug(image_bgr)
warped_bgr, warp_meta = omr.warp_sheet(image_bgr)

actual_corners = None
if warp_meta.get('frame_corners_original'):
    actual_corners = np.asarray(warp_meta['frame_corners_original'], dtype=np.float32)
elif warp_meta.get('paper_corners_original'):
    actual_corners = np.asarray(warp_meta['paper_corners_original'], dtype=np.float32)

frame_overlay = draw_quad(image_bgr, frame_debug['frame_corners'],
                          'Printed-frame candidate', (0, 220, 0))
paper_overlay = draw_quad(image_bgr, paper_debug['paper_corners'],
                          'Paper-contour candidate', (255, 0, 255))
actual_overlay = draw_quad(image_bgr, actual_corners,
                           f'Used by warp: {warp_meta.get("warp_method", "unknown")}',
                           (0, 255, 255))

save_image('02_frame_grayscale.png', frame_debug['gray'])
save_image('03_frame_gaussian_blur.png', frame_debug['blur'])
save_image('04_frame_canny_edges.png', frame_debug['edges'])
save_image('05_hough_line_segments_all.png', frame_debug['all_lines'])
save_image('06_hough_lines_orientation_filtered.png', frame_debug['oriented'])
save_image('07_printed_frame_corners.png', frame_overlay)
save_image('08_paper_contour_corners.png', paper_overlay)
save_image('09_actual_warp_quadrilateral.png', actual_overlay)
save_image('10_paper_otsu_threshold.png', paper_debug['otsu'])
save_image('11_paper_morphological_close.png', paper_debug['closed'])
save_image('12_paper_morphological_open.png', paper_debug['opened'])
save_image('13_perspective_corrected_sheet.png', warped_bgr)

print(json.dumps(warp_meta, indent=2))
print(f"Orientation-filtered segments: {frame_debug['horizontal_count']} horizontal, "
      f"{frame_debug['vertical_count']} vertical")

show_gallery([
    ('Canny edge map', frame_debug['edges']),
    ('All Hough segments', frame_debug['all_lines']),
    ('Orientation-filtered segments', frame_debug['oriented']),
    ('Actual quadrilateral used', actual_overlay),
    ('Paper fallback: Otsu + morphology', paper_debug['opened']),
    ('Perspective-corrected sheet', warped_bgr),
], columns=2)

# Stage 3 — Printed-Region Detection

The implementation applies illumination normalization, adaptive inverse thresholding, directional morphology, connected components, line-fragment merging, and geometric section recovery.

In [ ]:
def printed_line_debug(warped_bgr: np.ndarray) -> Dict[str, Any]:
    gray = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    normalized = omr._normalize_lighting(gray, sigma=35)
    threshold = cv2.adaptiveThreshold(
        normalized, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV, 31, 8
    )
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(45, w // 22), 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(35, h // 35)))
    h_open = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, h_kernel, iterations=1)
    v_open = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, v_kernel, iterations=1)
    horizontal = cv2.dilate(h_open, cv2.getStructuringElement(cv2.MORPH_RECT, (9, 1)), iterations=1)
    vertical = cv2.dilate(v_open, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 9)), iterations=1)
    h_lines = omr._component_lines(horizontal, 'h')
    v_lines = omr._component_lines(vertical, 'v')

    overlay = warped_bgr.copy()
    for line in h_lines:
        y = int(round(line['center']))
        cv2.line(overlay, (int(line['x1']), y), (int(line['x2']), y),
                 (255, 0, 0), 3, cv2.LINE_AA)
    for line in v_lines:
        x = int(round(line['center']))
        cv2.line(overlay, (x, int(line['y1'])), (x, int(line['y2'])),
                 (0, 140, 255), 3, cv2.LINE_AA)
    return {
        'gray': gray, 'normalized': normalized, 'threshold': threshold,
        'h_open': h_open, 'v_open': v_open,
        'horizontal': horizontal, 'vertical': vertical,
        'h_lines': h_lines, 'v_lines': v_lines, 'overlay': overlay,
    }


line_debug = printed_line_debug(warped_bgr)
layout_initial = omr.detect_sheet_layout(warped_bgr)
roi_overlay_initial = draw_roi_overlay(warped_bgr, layout_initial)

save_image('14_warped_grayscale.png', line_debug['gray'])
save_image('15_illumination_normalized.png', line_debug['normalized'])
save_image('16_adaptive_inverse_threshold.png', line_debug['threshold'])
save_image('17_horizontal_opening_mask.png', line_debug['h_open'])
save_image('18_vertical_opening_mask.png', line_debug['v_open'])
save_image('19_horizontal_line_mask_dilated.png', line_debug['horizontal'])
save_image('20_vertical_line_mask_dilated.png', line_debug['vertical'])
save_image('21_recovered_printed_lines.png', line_debug['overlay'])
save_image('22_detected_regions_overlay_initial.png', roi_overlay_initial)

print(json.dumps(layout_initial, indent=2))
show_gallery([
    ('Illumination-normalized sheet', line_debug['normalized']),
    ('Adaptive inverse threshold', line_debug['threshold']),
    ('Horizontal line mask', line_debug['horizontal']),
    ('Vertical line mask', line_debug['vertical']),
    ('Recovered printed lines', line_debug['overlay']),
    ('Detected/fallback ROIs', roi_overlay_initial),
], columns=2)

## Run the real end-to-end detector

This cell calls the public `detect_omr_bubbles()` function. Its result includes the final answer-ROI retry logic and is used by all later stages.

In [ ]:
result = omr.detect_omr_bubbles(image_bgr, expected_questions=EXPECTED_QUESTIONS)
metadata = result.get('metadata', {})
layout = result.get('layout', {})
answers = result.get('answers', {})
student_id = result.get('student_id', {})
test_id = result.get('test_id', {})

roi_overlay_final = draw_roi_overlay(warped_bgr, layout)
save_image('23_detected_regions_overlay_final.png', roi_overlay_final)

print('Warp method:', metadata.get('warp_method'))
print('Layout method:', metadata.get('layout_method'))
print('Detected questions:', answers.get('question_count', 0))
print('Rows per answer group:', answers.get('rows_per_group', []))
print('Answer bubbles:', answers.get('bubble_count', 0))
print('Student ID bubbles:', student_id.get('bubble_count', 0))
print('Test ID bubbles:', test_id.get('bubble_count', 0))
print('Final region sources:', layout.get('source', {}))

# Stage 4 — Bubble Candidate Detection

- Answer area: contour-based candidate detection.
- Student ID and Test ID: Hough-circle candidate detection.

In [ ]:
def draw_candidate_overlay(roi_bgr, candidates, color, title):
    out = roi_bgr.copy()
    for c in candidates:
        x, y, r = int(round(c.x)), int(round(c.y)), int(round(c.r))
        cv2.circle(out, (x, y), r, color, 2, cv2.LINE_AA)
        cv2.circle(out, (x, y), 1, color, -1)
    cv2.putText(out, f'{title}: {len(candidates)} candidates', (12, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2, cv2.LINE_AA)
    return out


def answer_candidate_debug(gray_roi):
    normalized = omr._normalize_lighting(gray_roi, sigma=25)
    blurred = cv2.GaussianBlur(normalized, (3, 3), 0)
    threshold = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 31, 7
    )
    contours, _ = cv2.findContours(threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    all_contours = cv2.cvtColor(gray_roi, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(all_contours, contours, -1, (170, 170, 170), 1)
    accepted = omr._contour_circle_candidates(gray_roi)
    accepted_overlay = draw_candidate_overlay(
        cv2.cvtColor(gray_roi, cv2.COLOR_GRAY2BGR), accepted,
        (0, 200, 0), 'Accepted contour circles'
    )
    return {
        'normalized': normalized, 'blurred': blurred, 'threshold': threshold,
        'contours': contours, 'all_contours': all_contours,
        'accepted': accepted, 'accepted_overlay': accepted_overlay,
    }


def id_candidate_debug(gray_roi, title):
    normalized = omr._normalize_lighting(gray_roi, sigma=17)
    equalized = cv2.equalizeHist(normalized)
    median = cv2.medianBlur(equalized, 3)
    accepted = omr._hough_circle_candidates(gray_roi)
    overlay = draw_candidate_overlay(
        cv2.cvtColor(gray_roi, cv2.COLOR_GRAY2BGR), accepted,
        (0, 140, 255), title
    )
    return {'normalized': normalized, 'equalized': equalized,
            'median': median, 'accepted': accepted, 'overlay': overlay}


warped_gray = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2GRAY)
answer_roi = answers.get('roi', layout.get('answers'))
student_roi = student_id.get('roi', layout.get('student_id'))
test_roi = test_id.get('roi', layout.get('test_id'))

answer_debug = answer_candidate_debug(crop_roi(warped_gray, answer_roi))
student_debug = id_candidate_debug(crop_roi(warped_gray, student_roi), 'Student ID Hough circles')
test_debug = id_candidate_debug(crop_roi(warped_gray, test_roi), 'Test ID Hough circles')

save_image('24_answer_candidate_normalized.png', answer_debug['normalized'])
save_image('25_answer_candidate_blurred.png', answer_debug['blurred'])
save_image('26_answer_candidate_threshold.png', answer_debug['threshold'])
save_image('27_answer_all_external_contours.png', answer_debug['all_contours'])
save_image('28_answer_accepted_contour_candidates.png', answer_debug['accepted_overlay'])
save_image('29_student_id_equalized.png', student_debug['equalized'])
save_image('30_student_id_median_blur.png', student_debug['median'])
save_image('31_student_id_hough_candidates.png', student_debug['overlay'])
save_image('32_test_id_equalized.png', test_debug['equalized'])
save_image('33_test_id_median_blur.png', test_debug['median'])
save_image('34_test_id_hough_candidates.png', test_debug['overlay'])

print(f"Answer contour candidates: {len(answer_debug['accepted'])}")
print(f"Student ID Hough candidates: {len(student_debug['accepted'])}")
print(f"Test ID Hough candidates: {len(test_debug['accepted'])}")

show_gallery([
    ('Answer adaptive threshold', answer_debug['threshold']),
    ('Accepted answer contour candidates', answer_debug['accepted_overlay']),
    ('Student ID equalized', student_debug['equalized']),
    ('Student ID Hough candidates', student_debug['overlay']),
    ('Test ID equalized', test_debug['equalized']),
    ('Test ID Hough candidates', test_debug['overlay']),
], columns=2)

# Stage 5 — Grid Inference and Bubble Assignment

The notebook marks final answer-grid positions as:

- **Green:** supported by a nearby detected contour.
- **Magenta:** reconstructed mainly from regular grid geometry.

It also exports the inferred Student ID and Test ID grids.

In [ ]:
def to_global_candidates(local_candidates, roi):
    x1, y1, _, _ = map(int, roi)
    return [(float(c.x) + x1, float(c.y) + y1, float(c.r)) for c in local_candidates]


def has_candidate_support(x, y, candidates, tolerance=9.0):
    return any(math.hypot(x - cx, y - cy) <= tolerance for cx, cy, _ in candidates)


def draw_answer_grid_debug(warped_bgr, answer_result, raw_candidates):
    out = warped_bgr.copy()
    for x, y, r in raw_candidates:
        cv2.circle(out, (int(round(x)), int(round(y))), max(2, int(round(r))),
                   (150, 150, 150), 1, cv2.LINE_AA)

    x_groups = answer_result.get('x_groups', [])
    rows_by_group = answer_result.get('y_rows_by_group', [])
    for idx, xs in enumerate(x_groups):
        rows = rows_by_group[idx] if idx < len(rows_by_group) else []
        if not rows:
            continue
        y_min, y_max = int(min(rows)), int(max(rows))
        for option_idx, x in enumerate(xs):
            cv2.line(out, (int(round(x)), y_min), (int(round(x)), y_max),
                     (255, 170, 0), 1, cv2.LINE_AA)
            cv2.putText(out, omr.OPTIONS[option_idx], (int(round(x)) - 5, max(20, y_min - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 170, 0), 1, cv2.LINE_AA)
        for y in rows:
            cv2.line(out, (int(min(xs)), int(round(y))), (int(max(xs)), int(round(y))),
                     (180, 0, 180), 1, cv2.LINE_AA)

    for b in answer_result.get('bubbles', []):
        x, y, r = float(b['x']), float(b['y']), int(round(float(b['r'])))
        color = (0, 190, 0) if has_candidate_support(x, y, raw_candidates) else (200, 0, 200)
        cv2.circle(out, (int(round(x)), int(round(y))), r, color, 2, cv2.LINE_AA)
        if b.get('option') == 'A':
            cv2.putText(out, str(b.get('question')), (int(round(x)) - 30, int(round(y)) + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 200), 1, cv2.LINE_AA)
    return out


def draw_digit_grid_debug(warped_bgr, section, local_candidates, title):
    out = warped_bgr.copy()
    roi = section.get('roi')
    if not roi:
        return out
    x1, y1, _, _ = map(int, roi)
    for c in local_candidates:
        cv2.circle(out, (int(round(c.x + x1)), int(round(c.y + y1))), int(round(c.r)),
                   (150, 150, 150), 1, cv2.LINE_AA)
    for b in section.get('bubbles', []):
        cv2.circle(out, (int(round(float(b['x']))), int(round(float(b['y'])))),
                   int(round(float(b['r']))), (255, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(out, title, (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255, 0, 0), 2, cv2.LINE_AA)
    return out


answer_raw_global = to_global_candidates(answer_debug['accepted'], answer_roi)
answer_grid_overlay = draw_answer_grid_debug(warped_bgr, answers, answer_raw_global)
student_grid_overlay = draw_digit_grid_debug(
    warped_bgr, student_id, student_debug['accepted'],
    'Student ID: raw Hough circles + inferred grid'
)
test_grid_overlay = draw_digit_grid_debug(
    warped_bgr, test_id, test_debug['accepted'],
    'Test ID: raw Hough circles + inferred grid'
)

save_image('35_answer_grid_inference_full_sheet.png', answer_grid_overlay)
save_image('36_answer_grid_inference_crop.png', crop_roi(answer_grid_overlay, answer_roi))
save_image('37_student_id_grid_inference.png', crop_roi(student_grid_overlay, student_roi))
save_image('38_test_id_grid_inference.png', crop_roi(test_grid_overlay, test_roi))

rows_per_group = answers.get('rows_per_group', [])
if rows_per_group:
    grid_summary = pd.DataFrame({
        'answer_group': range(1, len(rows_per_group) + 1),
        'detected_rows': rows_per_group,
    })
    save_dataframe(grid_summary, 'answer_rows_per_group.csv')
    display(grid_summary)

print('Answer x groups:', answers.get('x_groups', []))
print('Rows per group:', rows_per_group)
print('Question count:', answers.get('question_count', 0))

show_gallery([
    ('Answer grid: candidates and inferred positions', crop_roi(answer_grid_overlay, answer_roi)),
    ('Student ID inferred grid', crop_roi(student_grid_overlay, student_roi)),
    ('Test ID inferred grid', crop_roi(test_grid_overlay, test_roi)),
], columns=1, width=11, height=6)

# Stage 6 — Fill Measurement and Mark Confidence

The detector samples the inner `70%` of the bubble radius:

\[
\text{fill score} = \frac{255 - \text{mean interior intensity}}{255}
\]

The interpretation compares the top score, second-highest score, and median baseline for the same question.

In [ ]:
answer_reading = omr.read_answer_choices(answers)
responses_df = pd.DataFrame(answer_reading.get('responses', []))
answer_bubbles_df = pd.DataFrame(answers.get('bubbles', []))

if responses_df.empty or answer_bubbles_df.empty:
    raise RuntimeError('No answer rows are available for fill-score visualization.')

if EXAMPLE_QUESTION is None:
    selected_rows = responses_df[responses_df['status'] == 'selected']
    if not selected_rows.empty:
        example_question = int(selected_rows.sort_values('confidence', ascending=False).iloc[0]['question'])
    else:
        example_question = int(responses_df.iloc[0]['question'])
else:
    example_question = int(EXAMPLE_QUESTION)

question_bubbles = answer_bubbles_df[
    answer_bubbles_df['question'] == example_question
].sort_values('option_index').copy()
if question_bubbles.empty:
    raise ValueError(f'Question {example_question} was not detected.')

question_response = responses_df[
    responses_df['question'] == example_question
].iloc[0].to_dict()

mask_overlay = warped_bgr.copy()
for _, b in question_bubbles.iterrows():
    x, y, r = int(round(float(b['x']))), int(round(float(b['y']))), int(round(float(b['r'])))
    inner_r = max(3, int(round(r * 0.70)))
    cv2.circle(mask_overlay, (x, y), r, (150, 150, 150), 2, cv2.LINE_AA)
    cv2.circle(mask_overlay, (x, y), inner_r, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(mask_overlay, str(b['option']), (x - 5, y - r - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

x_min = max(0, int(question_bubbles['x'].min() - 45))
x_max = min(warped_bgr.shape[1], int(question_bubbles['x'].max() + 45))
y_center = int(question_bubbles['y'].median())
y_min = max(0, y_center - 45)
y_max = min(warped_bgr.shape[0], y_center + 45)
mask_crop = mask_overlay[y_min:y_max, x_min:x_max]
save_image(f'39_question_{example_question:03d}_inner_fill_masks.png', mask_crop)

plt.figure(figsize=(8, 5))
plt.bar(question_bubbles['option'], question_bubbles['fill_score'])
plt.axhline(question_response['baseline_fill_score'], linestyle='--', linewidth=1.5,
            label=f"Median baseline = {question_response['baseline_fill_score']:.3f}")
plt.ylim(0, max(0.55, float(question_bubbles['fill_score'].max()) + 0.08))
plt.xlabel('Answer option')
plt.ylabel('Fill score')
plt.title(f"Question {example_question}: status={question_response['status']}, "
          f"answer={question_response['detected_answer']}")
plt.legend()
plt.tight_layout()
fill_plot_path = OUTPUT_DIR / f'40_question_{example_question:03d}_fill_scores.png'
plt.savefig(fill_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print(f'Saved: {fill_plot_path.name}')

display(question_bubbles[['question', 'option', 'fill_score']])
display(pd.DataFrame([question_response]))
print(f"top={question_response['top_fill_score']:.4f}, "
      f"second={question_response['second_fill_score']:.4f}, "
      f"baseline={question_response['baseline_fill_score']:.4f}, "
      f"confidence={question_response['confidence']:.4f}, "
      f"status={question_response['status']}")
show_image(mask_crop, f'Question {example_question} — inner 70% sampling masks', figsize=(12, 3))

# Stage 7 — Student/Test ID Decoding and Answer Grading

When no answer key is supplied, IDs and answer statuses are still decoded, but no percentage score is produced.

In [ ]:
def parse_answer_key_text(text: str, question_count: int) -> Dict[int, str]:
    text = text.strip()
    if not text:
        return {}
    try:
        loaded = json.loads(text)
        if isinstance(loaded, dict):
            return {int(q): str(a).strip().upper() for q, a in loaded.items()
                    if str(a).strip().upper() in omr.OPTIONS}
        if isinstance(loaded, list):
            return {i: str(a).strip().upper() for i, a in enumerate(loaded, start=1)
                    if str(a).strip().upper() in omr.OPTIONS}
    except json.JSONDecodeError:
        pass
    pairs = re.findall(r'(?m)(\d+)\s*[:=,.)-]\s*([A-Ea-e])\b', text)
    if pairs:
        return {int(q): a.upper() for q, a in pairs}
    tokens = re.findall(r'[A-Ea-e]', text)
    return {i: a.upper() for i, a in enumerate(tokens[:question_count], start=1)}


question_count = int(answers.get('question_count', 0))
if ANSWER_KEY_TEXT.strip():
    answer_key = parse_answer_key_text(ANSWER_KEY_TEXT, question_count)
elif USE_DEMO_KEY:
    answer_key = {q: omr.OPTIONS[(q - 1) % len(omr.OPTIONS)]
                  for q in range(1, question_count + 1)}
else:
    answer_key = omr.normalize_answer_key(ANSWER_KEY)

grading = omr.grade_omr_result(result, answer_key=answer_key)
result['grading'] = grading
student_reading = grading['identity']['student_id']
test_reading = grading['identity']['test_id']
summary = grading['summary']

print('Student ID:', student_reading.get('value') or 'No grid')
print('Student ID complete:', student_reading.get('complete'))
print('Test ID:', test_reading.get('value') or 'No grid')
print('Test ID complete:', test_reading.get('complete'))
print('Answer-key coverage:', len(answer_key), '/', question_count)
print(json.dumps(summary, indent=2))

question_review_df = pd.DataFrame(grading.get('questions', []))
student_positions_df = pd.DataFrame(student_reading.get('positions', []))
test_positions_df = pd.DataFrame(test_reading.get('positions', []))

if not question_review_df.empty:
    save_dataframe(question_review_df, 'question_by_question_grading.csv')
    display(question_review_df[[
        'question', 'detected_answer', 'correct_answer', 'outcome',
        'status', 'confidence', 'top_fill_score', 'second_fill_score'
    ]].head(30))
if not student_positions_df.empty:
    save_dataframe(student_positions_df, 'student_id_decoding.csv')
    display(student_positions_df)
if not test_positions_df.empty:
    save_dataframe(test_positions_df, 'test_id_decoding.csv')
    display(test_positions_df)

In [ ]:
def draw_decoded_id_crop(warped_bgr, section, decoded, title):
    roi = section.get('roi')
    if not roi:
        return np.zeros((100, 300, 3), dtype=np.uint8)
    crop = crop_roi(warped_bgr, roi)
    x1, y1, _, _ = map(int, roi)
    status_by_position = {int(item['position']): item for item in decoded.get('positions', [])}
    for b in section.get('bubbles', []):
        position = int(b['digit_position'])
        value = int(b['value'])
        item = status_by_position.get(position, {})
        x = int(round(float(b['x']))) - x1
        y = int(round(float(b['y']))) - y1
        r = int(round(float(b['r'])))
        color, thickness = (150, 150, 150), 1
        if item.get('status') == 'selected' and item.get('digit') == value:
            color, thickness = (0, 180, 0), 3
        elif item.get('status') in {'multiple', 'unclear'} and value in item.get('selected_values', []):
            color, thickness = (0, 165, 255), 3
        cv2.circle(crop, (x, y), r, color, thickness, cv2.LINE_AA)
    cv2.rectangle(crop, (0, 0), (crop.shape[1], 34), (255, 255, 255), -1)
    cv2.putText(crop, f"{title}: {decoded.get('value', '') or 'No grid'}", (8, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)
    return crop


student_decoded_crop = draw_decoded_id_crop(warped_bgr, student_id, student_reading, 'Student ID')
test_decoded_crop = draw_decoded_id_crop(warped_bgr, test_id, test_reading, 'Test ID')
save_image('41_student_id_decoded.png', student_decoded_crop)
save_image('42_test_id_decoded.png', test_decoded_crop)
show_gallery([
    ('Decoded Student ID', student_decoded_crop),
    ('Decoded Test ID', test_decoded_crop),
], columns=1, width=11, height=4)

# Stage 8 — Final Visualization, Metadata, and Exports

This stage saves the final detection overlay, grading overlay, JSON result, bubble tables, decoding tables, grading table, and output manifest.

In [ ]:
detection_overlay = omr.draw_detection_overlay(
    warped_bgr, result, draw_rois=True, draw_labels=True
)
grading_overlay = omr.draw_grading_overlay(
    warped_bgr, result, draw_rois=True, draw_labels=True,
    show_correct_answers=bool(answer_key)
)

save_image('43_final_detection_overlay.png', detection_overlay)
save_image('44_final_grading_overlay.png', grading_overlay)

json_path = OUTPUT_DIR / 'omr_result.json'
json_path.write_text(json.dumps(omr.make_json_safe(result), indent=2), encoding='utf-8')
print(f'Saved: {json_path.name}')

metadata_path = OUTPUT_DIR / 'metadata_and_layout.json'
metadata_path.write_text(json.dumps(omr.make_json_safe({
    'metadata': result.get('metadata', {}),
    'layout': result.get('layout', {}),
}), indent=2), encoding='utf-8')
print(f'Saved: {metadata_path.name}')

for section_name in ('answers', 'student_id', 'test_id'):
    df = pd.DataFrame(result.get(section_name, {}).get('bubbles', []))
    if not df.empty:
        save_dataframe(df, f'{section_name}_bubble_table.csv')

show_gallery([
    ('Original phone photograph', image_bgr),
    ('Perspective-corrected sheet', warped_bgr),
    ('Detection overlay', detection_overlay),
    ('Grading overlay', grading_overlay),
], columns=2, width=7, height=6)

# Output Manifest

Run this cell after all previous stages. It lists every generated visualization and table.

In [ ]:
manifest_rows = []
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        manifest_rows.append({
            'file': path.name,
            'type': path.suffix.lower().lstrip('.'),
            'size_kb': round(path.stat().st_size / 1024.0, 1),
        })
manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df)
manifest_path = OUTPUT_DIR / 'output_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
print(f'Saved manifest: {manifest_path}')
print(f'All outputs are available in: {OUTPUT_DIR.resolve()}')

## Recommended presentation assets

The most useful files are:

- `01_original_phone_photo.png`
- `04_frame_canny_edges.png`
- `06_hough_lines_orientation_filtered.png`
- `09_actual_warp_quadrilateral.png`
- `13_perspective_corrected_sheet.png`
- `16_adaptive_inverse_threshold.png`
- `19_horizontal_line_mask_dilated.png`
- `20_vertical_line_mask_dilated.png`
- `23_detected_regions_overlay_final.png`
- `28_answer_accepted_contour_candidates.png`
- `31_student_id_hough_candidates.png`
- `34_test_id_hough_candidates.png`
- `36_answer_grid_inference_crop.png`
- `39_question_..._inner_fill_masks.png`
- `40_question_..._fill_scores.png`
- `41_student_id_decoded.png`
- `42_test_id_decoded.png`
- `44_final_grading_overlay.png`

Use the same input photograph across the methodology slides so the audience can follow one sheet from the original phone image to the final decoded result.